In [8]:
import re, os
import pandas as pd
from bertopic import BERTopic

In [9]:
os.makedirs("../data/processed/", exist_ok=True)

In [2]:
df = pd.read_csv("../data/scraped_last_10_dates.csv")
df = df.dropna(subset=['text']).reset_index(drop=True)

docs = df['text'].tolist()

topic_model = BERTopic(language="english", calculate_probabilities=True)

topics, probabilities = topic_model.fit_transform(docs)

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 3034.04it/s]


In [5]:
df['Topic'] = topics
df['Probabilities'] = probabilities

print("Overview of Generated Topics:")
topic_info = topic_model.get_topic_info()
print(topic_info.head())

if len(topic_info) > 1:
    top_valid_topic = topic_info[topic_info['Topic'] != -1]['Topic'].iloc[0]
    
    print(f"\nKeywords defining Topic {top_valid_topic}:")
    print(topic_model.get_topic(top_valid_topic))


Overview of Generated Topics:
   Topic  Count                   Name  \
0     -1     10  -1_the_russian_and_in   

                                      Representation  \
0  [the, russian, and, in, of, ukrainian, forces,...   

                                 Representative_Docs  
0  [Toplines Russian Chief of the General Staff A...  


In [6]:
df.to_csv("../data/processed/scraped_with_topics.csv", index=False)

In [10]:
def extract_snippet(text):
    if not isinstance(text, str):
        return ""
    
    takeaways_match = re.search(r'(Key Takeaways.*?)(?=We do not report in detail|\Z)', text, re.IGNORECASE | re.DOTALL)
    if takeaways_match:
        return takeaways_match.group(1).strip()
    
    toplines_match = re.search(r'(Toplines.*?)(?=Key Takeaways|We do not report in detail|\Z)', text, re.IGNORECASE | re.DOTALL)
    if toplines_match:
        return toplines_match.group(1).strip()
    
    return text[:500].strip() + "..." if len(text) > 500 else text

df['snippet'] = df['text'].apply(extract_snippet)

df = df.drop(columns=['text'])

df.to_csv("../data/processed/scraped_with_topics_snippets_only.csv", index=False)